In [1]:
import imaplib
import getpass
import sys
import re
from datetime import datetime, date, time, timedelta
from email import message_from_bytes, policy
from email.utils import parsedate_to_datetime
from html.parser import HTMLParser

In [2]:
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

IMAP_SERVER = "imap.gmail.com"
IMAP_PORT = 993

CATEGORIES = {
    "Primary":    "category:primary",
    "Promotions": "category:promotions",
    "Social":     "category:social",
    "Updates":    "category:updates",
    "Forums":     "category:forums",
}

# Turnimg emails from HTML to readble texts


In [3]:
class TextExtractor(HTMLParser):
    BLOCK_TAGS = {"br", "p", "div", "tr", "li", "h1", "h2", "h3", "h4", "h5", "h6"}

    def __init__(self):
        super().__init__()
        self.parts = []
        self.skip = 0

    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self.skip += 1
        elif tag in self.BLOCK_TAGS:
            self.parts.append("\n")

    def handle_endtag(self, tag):
        if tag in ("script", "style") and self.skip:
            self.skip -= 1
        elif tag in self.BLOCK_TAGS:
            self.parts.append("\n")

    def handle_data(self, data):
        if not self.skip:
            self.parts.append(data)

    def get_text(self):
        return "".join(self.parts)


def clean_text(text):
    lines = [re.sub(r"[ \t\xa0]+", " ", ln).strip() for ln in text.splitlines()]
    text = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def get_body(msg):
    part = msg.get_body(preferencelist=("plain", "html"))
    if part is None:
        return "(No readable text content)"
    try:
        content = part.get_content()
    except Exception:
        return "(Could not decode content)"
    if part.get_content_type() == "text/html":
        extractor = TextExtractor()
        extractor.feed(content)
        content = extractor.get_text()
    return clean_text(content) or "(Empty message)"


def local_date(msg):
    raw = msg.get("Date")
    try:
        return parsedate_to_datetime(raw).astimezone().strftime("%d-%b-%Y %I:%M %p")
    except Exception:
        return str(raw)

# Finding for the date

In [4]:
def ask_date(prompt):
    while True:
        text = input(prompt).strip()
        if text == "":
            return date.today()
        try:
            return datetime.strptime(text, "%Y-%m-%d").date()
        except ValueError:
            print("  Invalid date. Please use the format YYYY-MM-DD, e.g. 2026-10-10")


def to_epoch(d):
    return int(datetime.combine(d, time.min).timestamp())

# Searching through the gmail and returning the UID's

In [5]:
def search_uids(mail, extra_query, start_date, end_date):
    after = to_epoch(start_date)
    before = to_epoch(end_date + timedelta(days=1))
    query = f"{extra_query} after:{after} before:{before}".strip()
    status, data = mail.uid("SEARCH", "X-GM-RAW", f'"{query}"')
    if status != "OK":
        raise RuntimeError("Search failed on the mail server.")
    if not data or not data[0]:
        return []
    return data[0].split()


def fetch_message(mail, uid):
    # BODY.PEEK[] reads the email without marking it as read
    status, data = mail.uid("FETCH", uid, "(BODY.PEEK[])")
    if status != "OK" or not data or not isinstance(data[0], tuple):
        return None
    return message_from_bytes(data[0][1], policy=policy.default)


# Going through the 'main' part

In [6]:
def main():
    email_address = input("Enter your Gmail address: ").strip()
    app_password = getpass.getpass("Enter your Gmail App Password: ").replace(" ", "")

    start_date = ask_date("Enter the date (YYYY-MM-DD) or press Enter for today: ")

    print("\nWhich emails?")
    print("  1 - Only that one day")
    print("  2 - From that date until today")
    choice = input("Enter 1 or 2: ").strip()

    if choice == "2":
        end_date = date.today()
        label = f"{start_date} to {end_date}"
    else:
        end_date = start_date
        label = f"{start_date}"

    if end_date < start_date:
        print("\nThe date cannot be in the future.")
        return

    print("\nWhat do you want to see?")
    print("  1 - Counts only")
    print("  2 - Counts + list of emails (sender, subject, time)")
    print("  3 - Counts + full content of every email")
    view = input("Enter 1, 2 or 3: ").strip()
    if view not in ("1", "2", "3"):
        view = "1"

    mail = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
    try:
        mail.login(email_address, app_password)
        mail.select("INBOX", readonly=True)  # read-only

        print(f"\nFetching emails ({label})... please wait.\n")

        #fetching the emails as per the required date range
        total_uids = search_uids(mail, "", start_date, end_date)
        matches = {uid: [] for uid in total_uids}

        # Each email gets segregated as per their body in the email
        for name, query in CATEGORIES.items():
            for uid in search_uids(mail, query, start_date, end_date):
                if uid in matches:
                    matches[uid].append(name)

        #Each email goes to exactly one place classified as categories
        sections = {name: [] for name in CATEGORIES}
        sections["Unclear"] = []          #emails that have matched more than one tab
        sections["Uncategorized"] = []    #emails that have matched no tab
        for uid, found in matches.items():
            if len(found) == 1:
                sections[found[0]].append(uid)
            elif len(found) > 1:
                sections["Unclear"].append(uid)
            else:
                sections["Uncategorized"].append(uid)

        for name in sections:
            sections[name].sort(key=int, reverse=True)  #fetching the newest first

        #showing the details as per the choice
        if view in ("2", "3"):
            for name, uids in sections.items():
                if name in ("Unclear", "Uncategorized") and not uids:
                    continue
                print("=" * 70)
                print(f"  {name.upper()}  -  {len(uids)} email(s)")
                print("=" * 70)
                for number, uid in enumerate(uids, start=1):
                    msg = fetch_message(mail, uid)
                    if msg is None:
                        print(f"\n[{number}] (Could not read this email)")
                        continue
                    print(f"\n[{number}] From   : {msg.get('From', '(unknown)')}")
                    print(f"    Subject: {msg.get('Subject', '(no subject)')}")
                    print(f"    Date   : {local_date(msg)}")
                    if name == "Unclear":
                        print(f"    Matched: {', '.join(matches[uid])}")
                    if view == "3":
                        print("    Content:")
                        print("    " + "-" * 40)
                        for line in get_body(msg).splitlines():
                            print("    " + line)
                        print("    " + "-" * 40)
                print()

        total = len(total_uids)
        counted = sum(len(v) for v in sections.values())

        print("=" * 70)
        print(f"  SUMMARY ({label})")
        print("=" * 70)
        for name, uids in sections.items():
            if name in ("Unclear", "Uncategorized") and not uids:
                continue
            print(f"  {name:<14}: {len(uids)}")
        print("  " + "-" * 20)
        print(f"  {'Total':<14}: {total}")
        if counted == total:
            print("  Check: all sections add up to the total.")
        else:
            print("  Warning: sections do not add up to the total.")
        if sections["Unclear"]:
            print("\n  'Unclear' = emails that Gmail's search matched to more than one tab.")
            print("  Use option 2 to see which tabs each one matched.")

    except imaplib.IMAP4.error:
        print("\nLogin failed. Check your email address and App Password.")
    except Exception as e:
        print(f"\nSomething went wrong: {e}")
    finally:
        try:
            mail.logout()
        except Exception:
            pass


if __name__ == "__main__":
    main()

Enter your Gmail address:  joykiran.260105@gmail.com
Enter your Gmail App Password:  ········
Enter the date (YYYY-MM-DD) or press Enter for today:  



Which emails?
  1 - Only that one day
  2 - From that date until today


Enter 1 or 2:  1



What do you want to see?
  1 - Counts only
  2 - Counts + list of emails (sender, subject, time)
  3 - Counts + full content of every email


Enter 1, 2 or 3:  1



Fetching emails (2026-10-16)... please wait.

  SUMMARY (2026-10-16)
  Primary       : 0
  Promotions    : 0
  Social        : 0
  Updates       : 0
  Forums        : 0
  Unclear       : 4
  --------------------
  Total         : 4
  Check: all sections add up to the total.

  'Unclear' = emails that Gmail's search matched to more than one tab.
  Use option 2 to see which tabs each one matched.
